### Notebook 03: Attempting Distributed ML with PySpark

In this notebook, we try to shift model training to the Databricks cluster using `pyspark.ml`, the classic Spark MLlib API. The goal is to run distributed training and cross-validation directly on remote compute, with MLflow logging along the way.

However, `pyspark.ml` depends on a Java-based `SparkContext`, which is not available in Databricks Connect. Since Connect uses a client-server model without an embedded JVM, the pipeline fails when trying to initialize objects like `StandardScaler` or `CrossValidator`.

This notebook shows those limitations in action. In the next one, we will introduce a workaround using a newer API.


In [ ]:
# Set up Databricks Connect session
from databricks.connect import DatabricksSession
import os
import config

# Pull connection values from config file
os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

# Initialize Spark session routed through Databricks Connect
spark = DatabricksSession.builder.getOrCreate()


In [ ]:
# Import PySpark ML components and supporting libraries
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

# MLflow and utility imports
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.exceptions import MlflowException
from pyspark.sql.functions import array, col
from datetime import datetime

In [ ]:
# Define catalog, schema, and table names for reading data and registering the model
CATALOG_NAME = "alexander_booth"
SCHEMA_NAME = "default"
TABLE_NAME = "breast_cancer_training_data"
MODEL_NAME = "breast_cancer_model"

In [ ]:
# Build resource URIs for the table and model
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "label"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Use current date for experiment naming
today = datetime.now().strftime("%Y%m%d")
experiment_name = f"breast_cancer_{today}"

In [ ]:
# Configure MLflow to use Unity Catalog as registry backend
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Define path and artifact location for experiment
experiment_path = f"/Users/alexander.booth@databricks.com/{experiment_name}"

# Log artifacts to a Volume
artifact_location = "dbfs:/Volumes/alexander_booth/default/mlflow_artifacts" 

# Create experiment if it doesn't exist
try:
    mlflow.create_experiment(name=experiment_path, artifact_location=artifact_location)
except MlflowException:
    pass

mlflow.set_experiment(experiment_path)

<Experiment: artifact_location='dbfs:/Volumes/alexander_booth/default/mlflow_artifacts', creation_time=1746587255667, experiment_id='2860937170952767', last_update_time=1746587862103, lifecycle_stage='active', name='/Users/alexander.booth@databricks.com/breast_cancer_20250506', tags={'mlflow.experiment.sourceName': '/Users/alexander.booth@databricks.com/breast_cancer_20250506',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'alexander.booth@databricks.com',
 'mlflow.ownerId': '8950113264559567'}>

In [ ]:
# Read raw training data from Unity Catalog
df_raw = spark.read.table(input_table).dropna()

# Convert features into a single array column (required by MLlib)
label_col = "label"
feature_cols = [c for c in df_raw.columns if c != label_col]

# Build vectorized dataframe
df_vectorized = df_raw.select(
    col(label_col).cast("double").alias("label"),
    array(*[col(c) for c in feature_cols]).alias("features")
)

# Split data into training and test sets
train_df, test_df = df_vectorized.randomSplit([0.8, 0.2], seed=42)
train_df.show(5)


+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[10.95, 21.35, 71...|
|  0.0|[11.08, 18.83, 73...|
|  0.0|[11.76, 18.14, 75...|
|  0.0|[11.8, 16.58, 78....|
|  0.0|[11.84, 18.7, 77....|
+-----+--------------------+
only showing top 5 rows


In [ ]:
import logging

# Silence the log streaming server warnings
logger = logging.getLogger("pyspark.ml.torch")
logger.setLevel(logging.FATAL)
logger.propagate = False  # prevent bubbling up to root logger

In [ ]:
# Attempt to use PySpark ML pipeline with Spark Connect
# This will fail because these components depend on a JVM-based SparkContext,
# which is not available when using Databricks Connect

scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
lr = LogisticRegression(numTrainWorkers=2, featuresCol="scaled_features")

# Construct ML pipeline
pipeline = Pipeline(stages=[scaler, lr])

# Basic hyperparameter grid
grid = ParamGridBuilder().addGrid(lr.maxIter, [2, 200]).build()

# Wrap pipeline in cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=grid,
    parallelism=2,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
)

# Start MLflow run
now = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"logistic-regression-{now}"

with mlflow.start_run(run_name=run_name) as run:

    # This call will raise an AssertionError because no JVM is attached
    model = cv.fit(train_df)

    # Evaluate and log metrics
    predictions = model.transform(test_df)
    binary_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    mlflow.log_metric("areaUnderROC", binary_eval.evaluate(predictions))

    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    mlflow.log_metric("accuracy", evaluator.evaluate(predictions))

    # Log best model params
    best_model = model.bestModel.stages[1]
    for param, value in best_model.extractParamMap().items():
        mlflow.log_param(param.name, value)

    # Log model with schema
    sample_input = test_df.limit(5).toPandas()
    sample_output = predictions.limit(5).toPandas()[["prediction"]]
    signature = infer_signature(sample_input, sample_output)

    mlflow.spark.log_model(model.bestModel, artifact_path="model", signature=signature)

    # Register model in Unity Catalog
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print(f"Run ID: {run_id}")
    print(f"Model URI: {model_uri}")

    mlflow.register_model(model_uri=model_uri, name=uc_model_name)


AssertionError: 

### ⚠️ Why this fails with Spark Connect

This example uses `pyspark.ml`, the classic Spark MLlib API, which relies on the JVM and requires an active `SparkContext`.

Databricks Connect uses a client-server model that runs Python code without an embedded JVM. As a result, components like `StandardScaler`, `Pipeline`, or `CrossValidator` from `pyspark.ml` cannot be used as they attempt to initialize Java objects that don’t exist in the client environment.

In the next notebook, we’ll use the newer `pyspark.ml.connect` API, which was designed specifically for Spark Connect and avoids these issues. It's not perfect yet, but it gets us much closer.

👉 [Learn more about distributed ML with Spark Connect](https://docs.databricks.com/aws/en/machine-learning/train-model/distributed-training/distributed-ml-for-spark-connect)
